# 04 - Polars e DuckDB

Alternativas modernas e performaticas ao Pandas.

## 1. Polars LazyFrame

In [ ]:
import polars as pl
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
npd = pd.DataFrame({
    'nome': [f'Func_{i}' for i in range(500)],
    'depto': rng.choice(['TI', 'RH', 'Financeiro', 'Vendas'], 500),
    'salario': rng.integers(4000, 15000, 500),
})
df_pl = pl.from_pandas(npd)
print(df_pl.head(5))

In [ ]:
result = (
    df_pl.lazy()
    .group_by('depto')
    .agg([
        pl.col('salario').mean().alias('media'),
        pl.col('salario').std().alias('desvio'),
        pl.len().alias('qtd'),
    ])
    .sort('media', descending=True)
    .collect()
)
print(result)

## 2. DuckDB - SQL direto no DataFrame

In [ ]:
import duckdb

conn = duckdb.connect()
result = conn.execute("""
    SELECT depto, AVG(salario)::INT AS media, COUNT(*) AS qtd
    FROM npd
    GROUP BY depto
    ORDER BY media DESC
""").fetchdf()
print(result)

## 3. DuckDB - SQL sobre CSV

In [ ]:
npd.to_csv('/tmp/exemplo_duckdb.csv', index=False)
result2 = conn.execute("""
    SELECT * FROM read_csv_auto('/tmp/exemplo_duckdb.csv')
    WHERE salario > 10000
    ORDER BY salario DESC
    LIMIT 10
""").fetchdf()
print(result2)

## 4. Performance

In [ ]:
import time

large = pd.DataFrame({'x': rng.standard_normal(1_000_000)})

t0 = time.perf_counter()
_ = large['x'].apply(lambda v: v ** 2)
pandas_t = time.perf_counter() - t0

t0 = time.perf_counter()
_ = pl.from_pandas(large)['x'] ** 2
polars_t = time.perf_counter() - t0

print(f'Pandas apply:  {pandas_t:.4f}s')
print(f'Polars vector: {polars_t:.4f}s')
print(f'Polars e {pandas_t/polars_t:.1f}x mais rapido')

## 5. Exercicio

Usando DuckDB, escreva uma query SQL que retorne o departmento com maior media salarial e o departmento com maior numero de funcionarios.

## Conclusao

Polars e DuckDB sao excelentes para exploracao rapida. Para dados maiores que a memoria, use PySpark.